# M2 IA Agente - Geração de Gráficos

Estamos muito felizes em tê-lo(a) aqui no curso de **IA Agente**! Neste **laboratório não avaliado**, e nos que se seguem ao longo do curso, você terá a oportunidade de experimentar exemplos de código que implementam os conceitos e padrões de projeto que você verá nas videoaulas.

Considere esses laboratórios como um **ambiente de testes**: um espaço seguro para praticar, onde você pode aprofundar sua compreensão dos conceitos do curso, ganhar confiança e se preparar para os exercícios avaliados que virão depois. Em cada laboratório, tente executar as células de código para ver alguns fluxos de trabalho agentes em ação e entender melhor como eles funcionam.

Em alguns pontos, você será incentivado(a) a fazer modificações no código, como alterar os prompts, testar com diferentes LLMs ou adicionar consultas adicionais ao fluxo de trabalho. Experimente para ver como suas alterações impactam o comportamento do fluxo de trabalho.

O mais importante é que os laboratórios não avaliados são uma oportunidade para aprender no seu próprio ritmo, enquanto adquire experiência prática com as ideias centrais da **IA Agente**. E lembre-se: você não está aprendendo sozinho! Se tiver alguma dúvida, fique à vontade para perguntar na
<a href="https://community.deeplearning.ai/c/course-q-a/agentic-ai/567" target="_blank">comunidade</a>

## 1. Introdução
### 1.1. Visão geral do laboratório

Neste laboratório não avaliado, você implementará o **padrão de reflexão** apresentado na videoaula em um fluxo de trabalho agentivo que gera visualizações de dados. Um Modelo de Linguagem Amplo (LLM) multimodal revisará o primeiro rascunho do gráfico, identificará possíveis melhorias — como tipo de gráfico, rótulos ou escolhas de cores — e, em seguida, reescreverá o código de geração do gráfico para produzir uma visualização mais eficaz.

No vídeo, Andrew apresentou o seguinte fluxo de trabalho para analisar as vendas de café. Você implementará isso em código aqui. As etapas que o fluxo de trabalho executará são:

1. **Gerar uma versão inicial (V1):**
Use um Modelo de Linguagem Amplo (LLM) para criar a primeira versão do código de plotagem. 2. **Executar o código e criar o gráfico:** Execute o código gerado e exiba o gráfico resultante. **(verifique todos os aspectos)**

3. **Reflita sobre o resultado:** Avalie o código e o gráfico usando um Modelo de Aprendizagem Linear (LLM) para detectar áreas de melhoria (por exemplo, clareza, precisão, design).

4. **Gerar e executar a versão aprimorada (V2):** Produza uma versão refinada do código de plotagem com base nas percepções da reflexão e renderize o gráfico aprimorado.

<img src='M2-UGL-2.png'>

### 🎯 1.2. Resultado de aprendizagem

Ao final deste laboratório, você terá implementado o padrão de reflexão em código e o utilizado para aprimorar uma visualização de dados.

## 2. Configuração: Inicializar o ambiente e o cliente

Nesta etapa, você importa as principais bibliotecas que darão suporte ao fluxo de trabalho:

- **`re`**: O módulo de expressões regulares do Python, que você usará para extrair trechos de código ou texto estruturado da saída do LLM.

- **`json`**: Fornece funções para ler e gravar JSON, úteis para lidar com respostas estruturadas retornadas pelo LLM.

- **`utils`**: Um módulo auxiliar personalizado fornecido para este laboratório. Ele inclui funções utilitárias para trabalhar com o conjunto de dados, gerar gráficos e exibir resultados em um formato limpo e legível.

In [ ]:
# Standard library imports
import re
import json

# Local helper module
import utils

### 2.1. Carregando o conjunto de dados

Vamos analisar os dados de vendas de café para ver quais informações estão contidas no arquivo.

In [ ]:
# Use this utils.py function to load the data into a dataframe
df = utils.load_and_prepare_data('coffee_sales.csv')

# Grab a random sample to display
utils.print_html(df.sample(n=5), title="Random Sample of Coffee Sales Data")

Você criará um fluxo de trabalho orientado a agentes que gera visualizações de dados a partir desse conjunto de dados, ajudando você a responder perguntas sobre as vendas de café da máquina de venda automática.

## 3. Construindo o pipeline

### 3.1 Etapa 1 — Gerar código para criar um gráfico (V1)

Nesta etapa, você solicitará a um modelo de lógica de aprendizado (LLM) que escreva um código Python que gere um gráfico em resposta a uma consulta do usuário sobre o conjunto de dados de café. O conjunto de dados inclui campos como `date`, `coffee_type`, `quantity` e `revenue`, e você passará esse esquema para o LLM para que ele saiba quais dados estão disponíveis.

A pergunta que você fará ao modelo é a mesma usada na aula:
**“Crie um gráfico comparando as vendas de café do primeiro trimestre de 2024 e 2025 usando os dados em coffee_sales.csv.”**

A saída do LLM será um código Python usando a biblioteca **matplotlib**. Em vez de exibir o gráfico diretamente, o código será escrito entre tags `<execute_python>` para que possa ser extraído e executado em etapas posteriores. Você aprenderá mais sobre essas tags no Módulo 3.

In [ ]:
def generate_chart_code(instruction: str, model: str, out_path_v1: str) -> str:
    """Generate Python code to make a plot with matplotlib using tag-based wrapping."""

    prompt = f"""
    You are a data visualization expert.

    Return your answer *strictly* in this format:

    <execute_python>
    # valid python code here
    </execute_python>

    Do not add explanations, only the tags and the code.

    The code should create a visualization from a DataFrame 'df' with these columns:
    - date (M/D/YY)
    - time (HH:MM)
    - cash_type (card or cash)
    - card (string)
    - price (number)
    - coffee_name (string)
    - quarter (1-4)
    - month (1-12)
    - year (YYYY)

    User instruction: {instruction}

    Requirements for the code:
    1. Assume the DataFrame is already loaded as 'df'.
    2. Use matplotlib for plotting.
    3. Add clear title, axis labels, and legend if needed.
    4. Save the figure as '{out_path_v1}' with dpi=300.
    5. Do not call plt.show().
    6. Close all plots with plt.close().
    7. Add all necessary import python statements

    Return ONLY the code wrapped in <execute_python> tags.
    """

    response = utils.get_response(model, prompt)
    return response

Agora, experimente a função e analise a resposta!

In [ ]:
# Generate initial code
code_v1 = generate_chart_code(
    instruction="Create a plot comparing Q1 coffee sales in 2024 and 2025 using the data in coffee_sales.csv.", 
    model="gpt-4o-mini", 
    out_path_v1="chart_v1.png"
)

utils.print_html(code_v1, title="LLM output with first draft code")

Ótimo! Você gerou um código Python para criar um gráfico!

Observe que o código está entre as tags `<execute_python>`. Essas tags facilitam a extração e execução automática do código na próxima etapa do fluxo de trabalho.

Não se preocupe com os detalhes agora — você aprenderá mais sobre como essas tags funcionam no **Módulo 3**.

### 3.2. Etapa 2 — Executar o código e criar o gráfico

Nesta etapa, você usará uma expressão regular para extrair o código Python gerado pelo LLM na etapa anterior (a parte escrita entre as tags `<execute_python>`). Após a extração, você executará esse código para gerar o **primeiro rascunho do gráfico**.

Veja como funciona:

1. **Extrair o código:**

Um padrão de expressão regular é usado para obter o código que está dentro das tags `<execute_python>`.

2. **Executar o código:**

O código extraído é executado em um contexto global predefinido, onde o DataFrame `df` já está disponível. Isso significa que seu código pode usar o `df` diretamente, sem precisar recarregar o conjunto de dados.

3. **Gerar o gráfico:**

Se o código for executado com sucesso, ele criará um gráfico e o salvará como `chart_v1.png`.

4. **Visualize o gráfico no notebook:**
O gráfico salvo é então exibido em linha usando `utils.print_html`, facilitando a revisão dos resultados.

Ao concluir esta etapa, você terá sua primeira versão da visualização (V1) pronta — um grande marco no fluxo de trabalho de reflexão!

In [ ]:
# Get the code within the <execute_python> tags
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v1)
if match:
    initial_code = match.group(1).strip()
    utils.print_html(initial_code, title="Extracted Code to Execute")
    exec_globals = {"df": df}
    exec(initial_code, exec_globals)

# If code run successfully, the file chart_v1.png should have been generated
utils.print_html(
    content="chart_v1.png",
    title="Generated Chart (V1)",
    is_image=True
)

### 3.3. Etapa 3 — Reflita sobre o resultado

O objetivo aqui é simular como um humano revisaria um primeiro rascunho de um gráfico — procurando pontos fortes, pontos fracos e áreas para melhoria.

Veja como funciona:

**1. Forneça o gráfico ao LLM:**
O gráfico gerado (chart_v1.png) é compartilhado com o LLM para que ele possa "ver" a visualização.

**2. Analise o gráfico visualmente:**
O LLM revisa elementos como clareza, rotulagem, precisão e legibilidade geral.

**3. Gere feedback:**
O LLM sugere melhorias — por exemplo, corrigir rótulos de eixos, ajustar o tipo de gráfico, aprimorar as escolhas de cores ou destacar legendas ausentes.

Ao fazer isso, você cria um ciclo de feedback inteligente em que o gráfico não é apenas produzido uma vez, mas ativamente criticado — preparando o terreno para uma segunda versão (V2) mais robusta.

In [ ]:
def reflect_on_image_and_regenerate(
    chart_path: str,
    instruction: str,
    model_name: str,
    out_path_v2: str,
    code_v1: str,  
) -> tuple[str, str]:
    """
    Critique the chart IMAGE and the original code against the instruction, 
    then return refined matplotlib code.
    Returns (feedback, refined_code_with_tags).
    Supports OpenAI and Anthropic (Claude).
    """
    media_type, b64 = utils.encode_image_b64(chart_path)
    

    prompt = f"""
    You are a data visualization expert.
    Your task: critique the attached chart and the original code against the given instruction,
    then return improved matplotlib code.

    Original code (for context):
    {code_v1}

    OUTPUT FORMAT (STRICT):
    1) First line: a valid JSON object with ONLY the "feedback" field.
    Example: {{"feedback": "The legend is unclear and the axis labels overlap."}}

    2) After a newline, output ONLY the refined Python code wrapped in:
    <execute_python>
    ...
    </execute_python>

    3) Import all necessary libraries in the code. Don't assume any imports from the original code.

    HARD CONSTRAINTS:
    - Do NOT include Markdown, backticks, or any extra prose outside the two parts above.
    - Use pandas/matplotlib only (no seaborn).
    - Assume df already exists; do not read from files.
    - Save to '{out_path_v2}' with dpi=300.
    - Always call plt.close() at the end (no plt.show()).
    - Include all necessary import statements.

    Schema (columns available in df):
    - date (M/D/YY)
    - time (HH:MM)
    - cash_type (card or cash)
    - card (string)
    - price (number)
    - coffee_name (string)
    - quarter (1-4)
    - month (1-12)
    - year (YYYY)

    Instruction:
    {instruction}
    """


    # In case the name is "Claude" or "Anthropic", use the safe helper
    lower = model_name.lower()
    if "claude" in lower or "anthropic" in lower:
        # ✅ Use the safe helper that joins all text blocks and adds a system prompt
        content = utils.image_anthropic_call(model_name, prompt, media_type, b64)
    else:
        content = utils.image_openai_call(model_name, prompt, media_type, b64)

    # --- Parse ONLY the first JSON line (feedback) ---
    lines = content.strip().splitlines()
    json_line = lines[0].strip() if lines else ""

    try:
        obj = json.loads(json_line)
    except Exception as e:
        # Fallback: try to capture the first {...} in all the content
        m_json = re.search(r"\{.*?\}", content, flags=re.DOTALL)
        if m_json:
            try:
                obj = json.loads(m_json.group(0))
            except Exception as e2:
                obj = {"feedback": f"Failed to parse JSON: {e2}", "refined_code": ""}
        else:
            obj = {"feedback": f"Failed to find JSON: {e}", "refined_code": ""}

    # --- Extract refined code from <execute_python>...</execute_python> ---
    m_code = re.search(r"<execute_python>([\s\S]*?)</execute_python>", content)
    refined_code_body = m_code.group(1).strip() if m_code else ""
    refined_code = utils.ensure_execute_python_tags(refined_code_body)

    feedback = str(obj.get("feedback", "")).strip()
    return feedback, refined_code



Observe que o modelo está instruído a retornar sua resposta em **formato JSON**.

- JSON é um formato leve e estruturado (pares chave-valor) que facilita a análise programática da saída do LLM.

- Aqui, exigimos dois campos:

- **`feedback`**: uma breve crítica ao gráfico atual.

- **`refined_code`**: um trecho de código Python aprimorado, envolto em tags `<execute_python>`.

Também incluímos uma seção de **“restrições”** no prompt. Essas regras (por exemplo, usar apenas matplotlib, salvar o arquivo em um caminho específico, chamar `plt.close()` ao final) ajudam o modelo a gerar um código consistente e executável que se adequa ao fluxo de trabalho. Sem essas restrições, a saída pode variar muito ou incluir formatação indesejada.

### 3.4 Etapa 4 — Gerar e Executar a Versão Aprimorada (V2)

Nesta etapa final, é hora de gerar e executar a versão aprimorada do gráfico (V2).

Após executar a célula, você verá **tanto a reflexão escrita pelo LLM** (explicando o que precisava ser aprimorado) **quanto o novo código gerado**. O novo código será então executado para produzir o gráfico atualizado.

In [ ]:
# Generate feedback alongside reflected code
feedback, code_v2 = reflect_on_image_and_regenerate(
    chart_path="chart_v1.png",            
    instruction="Create a plot comparing Q1 coffee sales in 2024 and 2025 using the data in coffee_sales.csv.", 
    model_name="o4-mini",
    out_path_v2="chart_v2.png",
    code_v1=code_v1,   # pass in the original code for context        
)

utils.print_html(feedback, title="Feedback on V1 Chart")
utils.print_html(code_v2, title="Regenerated Code Output (V2)")

Agora você executará o código refinado retornado pela etapa de reflexão. O código dentro das tags `<execute_python>` é extraído, executado no conjunto de dados e usado para gerar o gráfico atualizado.

Se a execução for bem-sucedida, você verá a nova imagem (`chart_v2.png`) exibida abaixo como o **Gráfico Regenerado (V2)**.

In [ ]:
# Get the code within the <execute_python> tags
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v2)
if match:
    reflected_code = match.group(1).strip()
    exec_globals = {"df": df}
    exec(reflected_code, exec_globals)

# If code run successfully, the file chart_v2.png should have been generated
utils.print_html(
    content="chart_v2.png",
    title="Regenerated Chart (V2)",
    is_image=True
)

### 4. Juntando tudo — criando o fluxo de trabalho completo

Agora é hora de integrar tudo em um único fluxo de trabalho automatizado que o agente possa executar do início ao fim.

A função `run_workflow` conecta os componentes que você implementou anteriormente:

1) **Carregar e preparar dados** — via `utils.load_and_prepare_data(...)`.

2) **Gerar código V1** — com `generate_chart_code(...)`, que retorna o código matplotlib de rascunho inicial (envolvido em tags `<execute_python>`).

3) **Executar V1 imediatamente** — o fluxo de trabalho extrai o código entre as tags `<execute_python>` e o executa para produzir a primeira imagem do gráfico.

4) **Refletir e refinar** — `reflect_on_image_and_regenerate(...)` analisa a imagem V1 (e o código original) em relação à instrução, retornando um **feedback** conciso e o **código revisado (V2)**.

5) **Executar V2 imediatamente** — o código refinado é extraído e executado para gerar o gráfico aprimorado.

### O que este fluxo de trabalho aceita
- **`dataset_path`**: localização do arquivo CSV de entrada.

- **`user_instructions`**: a solicitação do gráfico (por exemplo, “Crie um gráfico comparando as vendas de café do primeiro trimestre de 2024 e 2025 usando os dados em coffee_sales.csv.”).

- **`generation_model`**: modelo usado para a geração inicial do código.

- **`reflection_model`**: modelo usado para a reflexão baseada em imagem e o refinamento do código.
- **`image_basename`**: nome base do arquivo para salvar as imagens dos gráficos (por exemplo, `chart_v1.png`, `chart_v2.png`).

> Observação: as etapas de execução do gráfico são intencionalmente **codificadas** para serem executadas logo após a geração/refinamento do código. Isso reflete o fluxo de trabalho da aula e garante que você veja a saída de cada versão antes de prosseguir.

In [ ]:
def run_workflow(
    dataset_path: str,
    user_instructions: str,
    generation_model: str,
    reflection_model: str,   
    image_basename: str = "chart",
):
    """
    End-to-end pipeline:
      1) load dataset
      2) generate V1 code
      3) execute V1 → produce chart_v1.png
      4) reflect on V1 (image + original code) → feedback + refined code
      5) execute V2 → produce chart_v2.png

    Returns a dict with all artifacts (codes, feedback, image paths).
    """
    # 0) Load dataset; utils handles parsing and feature derivations (e.g., year/quarter)
    df = utils.load_and_prepare_data(dataset_path)
    utils.print_html(df.sample(n=5), title="Random Sample of Dataset")

    # Paths to store charts
    out_v1 = f"{image_basename}_v1.png"
    out_v2 = f"{image_basename}_v2.png"

    # 1) Generate code (V1)
    utils.print_html("Step 1: Generating chart code (V1)… 📈")
    code_v1 = generate_chart_code(
        instruction=user_instructions,
        model=generation_model,
        out_path_v1=out_v1,
    )
    utils.print_html(code_v1, title="LLM output with first draft code (V1)")

    # 2) Execute V1 (hard-coded: extract <execute_python> block and run immediately)
    utils.print_html("Step 2: Executing chart code (V1)… 💻")
    match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v1)
    if match:
        initial_code = match.group(1).strip()
        exec_globals = {"df": df}
        exec(initial_code, exec_globals)
    utils.print_html(out_v1, is_image=True, title="Generated Chart (V1)")

    # 3) Reflect on V1 (image + original code) to get feedback and refined code (V2)
    utils.print_html("Step 3: Reflecting on V1 (image + code) and generating improvements… 🔁")
    feedback, code_v2 = reflect_on_image_and_regenerate(
        chart_path=out_v1,
        instruction=user_instructions,
        model_name=reflection_model,
        out_path_v2=out_v2,
        code_v1=code_v1,  # pass original code for context
    )
    utils.print_html(feedback, title="Reflection feedback on V1")
    utils.print_html(code_v2, title="LLM output with revised code (V2)")

    # 4) Execute V2 (hard-coded: extract <execute_python> block and run immediately)
    utils.print_html("Step 4: Executing refined chart code (V2)… 🖼️")
    match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v2)
    if match:
        reflected_code = match.group(1).strip()
        exec_globals = {"df": df}
        exec(reflected_code, exec_globals)
    utils.print_html(out_v2, is_image=True, title="Regenerated Chart (V2)")

    return {
        "code_v1": code_v1,
        "chart_v1": out_v1,
        "feedback": feedback,
        "code_v2": code_v2,
        "chart_v2": out_v2,
    }


### 4.2. Experimente o fluxo de trabalho

Agora é a sua vez de colocar o fluxo de trabalho completo em ação com o exemplo atualizado da aula.

- **Instruções de uso:**

“Crie um gráfico comparando as vendas de café do primeiro trimestre de 2024 e 2025 usando os dados do arquivo coffee_sales.csv.”

Ao executar o fluxo de trabalho com essas instruções, ele irá:
1. Gerar um código de rascunho inicial para criar o gráfico.

2. Executar esse código imediatamente para produzir a primeira versão do gráfico (V1).

3. Analisar o gráfico e o código original, gerando feedback e código revisado (V2).

4. Executar o código refinado para gerar o gráfico aprimorado (V2).

### Personalize e experimente

Após experimentar o exemplo acima, sinta-se à vontade para atualizar o parâmetro `user_instructions` com suas próprias instruções para o gráfico.

Lembre-se também de ajustar o `image_basename` para que cada execução salve seus resultados com um novo nome de arquivo — isso mantém seus gráficos organizados e evita sobrescrever saídas anteriores.

### Escolhendo modelos

Você pode combinar diferentes modelos para geração e reflexão. Por exemplo:
- Use um modelo rápido para a geração inicial de código (`gpt-4.1-mini` ou `gpt-3.5-turbo`).

- Use um modelo de raciocínio mais robusto para reflexão (`gpt-4.1` ou `claude-3-7-sonnet-latest`).

Essa flexibilidade permite explorar o equilíbrio entre velocidade e qualidade.

👉 **Chamada para ação:** Execute o fluxo de trabalho agora com as instruções de exemplo da aula. Em seguida, experimente com seus próprios prompts para ver como o agente se adapta!

In [ ]:
# Here, insert your updates
user_instructions="Create a plot comparing Q1 coffee sales in 2024 and 2025 using the data in coffee_sales.csv." # write your instruction here
generation_model="gpt-4o-mini"
reflection_model="o4-mini"
image_basename="drink_sales"

# Run the complete agentic workflow
_ = run_workflow(
    dataset_path="coffee_sales.csv",
    user_instructions=user_instructions,
    generation_model=generation_model,
    reflection_model=reflection_model,
    image_basename=image_basename
)

## 5. Conclusões Finais

Neste laboratório, você praticou o uso da reflexão para aprimorar a criação de gráficos.

Você aprendeu a:

* Gerar um gráfico inicial (V1).

* Analisá-lo criticamente e refiná-lo para uma versão melhor (V2).

* Automatizar todo o fluxo de trabalho com diferentes modelos.

A ideia principal: a reflexão ajuda você a criar visualizações mais claras, precisas e eficazes.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>Parabéns!</strong>

Você concluiu o laboratório sobre a criação de um **fluxo de trabalho de geração de gráficos agéticos**.

Ao longo do processo, **você** praticou a geração de gráficos, refletiu sobre a qualidade deles e os refinou para criar visualizações mais claras e eficazes.

Com essas habilidades, **você** está pronto para projetar pipelines agéticos que criam visualizações de dados automaticamente, mantendo-as precisas, explicáveis ​​e refinadas. 🌟

</div>